# Interactive Reprint Network Visualization

This notebook creates an interactive geographic network showing how articles move across space and time through reprinting.

**Features:**
- Geographic layout based on publisher locations
- Time shown as color gradient (earlier = cooler colors, later = warmer colors)
- Directed edges showing reprint relationships
- Interactive: click nodes to see details and highlight lineage
- Focus on selected reprint groups

**Visualization Design:**
- **Nodes** = Articles (positioned geographically)
- **Node color** = Publication date (gradient)
- **Node size** = Fixed (or could be customized)
- **Edges** = Reprint relationships (original → reprint)
- **Edge style** = By reprint type (solid/dashed/dotted)

## Step 1: Import Libraries

In [34]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime
import numpy as np
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time


## Step 2: Helper Functions

These functions will:
1. Geocode location names to coordinates
2. Parse dates and create time-based color scales
3. Build the network structure

In [35]:
def geocode_locations(df):
    """
    Convert location names to latitude/longitude coordinates.
    Creates a cache to avoid re-geocoding the same locations.
    """
    # Get unique locations
    unique_locations = df['publisher_location'].dropna().unique()
    
    print(f"Geocoding {len(unique_locations)} unique locations...")
    
    # Create geocoder with rate limiting
    geolocator = Nominatim(user_agent="lonewoman_reprint_network")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    
    # Cache coordinates
    location_coords = {}
    
    for location in unique_locations:
        try:
            result = geocode(location)
            if result:
                location_coords[location] = {
                    'lat': result.latitude,
                    'lon': result.longitude
                }
                print(f"  ✓ {location}: ({result.latitude:.2f}, {result.longitude:.2f})")
            else:
                print(f"  ✗ Could not geocode: {location}")
                location_coords[location] = {'lat': None, 'lon': None}
        except Exception as e:
            print(f"  ✗ Error geocoding {location}: {e}")
            location_coords[location] = {'lat': None, 'lon': None}
    
    return location_coords

def add_coordinates_to_df(df, location_coords):
    """
    Add latitude and longitude columns to dataframe based on publisher_location.
    """
    df['lat'] = df['publisher_location'].map(lambda x: location_coords.get(x, {}).get('lat'))
    df['lon'] = df['publisher_location'].map(lambda x: location_coords.get(x, {}).get('lon'))
    return df



def parse_dates_with_era_colors(df):
    """
    Convert dates to datetime and create color values based on 50-year eras.
    
    Era ranges:
    - 1800-1849: Color 1 (Blues)
    - 1850-1899: Color 2 (Greens)
    - 1900-1949: Color 3 (Yellows/Oranges)
    - 1950-1999: Color 4 (Reds)
    - 2000+:      Color 5 (Purples)
    
    Within each era, decades create lighter/darker shades.
    """
    
    # Convert to datetime
    df['date_dt'] = pd.to_datetime(df['date'], errors='coerce')
    
    # Extract year
    df['year'] = df['date_dt'].dt.year
    
    # Determine era (50-year range)
    def get_era(year):
        if pd.isna(year):
            return None
        elif year < 1850:
            return 1  # 1800-1849
        elif year < 1900:
            return 2  # 1850-1899
        elif year < 1950:
            return 3  # 1900-1949
        elif year < 2000:
            return 4  # 1950-1999
        else:
            return 5  # 2000+
    
    df['era'] = df['year'].apply(get_era)
    
    # Calculate decade within era (0.0 to 1.0)
    # This creates gradation within each 50-year range
    def get_decade_position(year, era):
        if pd.isna(year) or pd.isna(era):
            return 0.5
        
        # Get the start year of the era
        era_starts = {1: 1800, 2: 1850, 3: 1900, 4: 1950, 5: 2000}
        era_start = era_starts[era]
        
        # Calculate position within 50 years (0.0 = start, 1.0 = end)
        position = (year - era_start) / 50.0
        
        # Clamp to 0-1 range
        return max(0, min(1, position))
    
    df['decade_position'] = df.apply(
        lambda row: get_decade_position(row['year'], row['era']), 
        axis=1
    )
    
    # Create a combined color value
    # Format: era.decade_position
    # e.g., 1.0 = 1800 (era 1, start)
    # e.g., 1.5 = 1825 (era 1, middle)
    # e.g., 2.0 = 1850 (era 2, start)
    df['color_value'] = df['era'] + df['decade_position']
    
    return df


def create_custom_colorscale():
    """
    Create a custom colorscale with 5 distinct colors for 5 eras,
    with gradations within each era.
    
    Color ranges:
    1.0-1.99: Blues (1800-1849)
    2.0-2.99: Greens (1850-1899)
    3.0-3.99: Yellows/Oranges (1900-1949)
    4.0-4.99: Reds (1950-1999)
    5.0-5.99: Purples (2000+)
    """
    
    colorscale = [
        # Era 1: 1800-1849 (Blues)
        [0.0/5, 'rgb(8, 48, 107)'],      # 1800: Dark blue
        [0.5/5, 'rgb(33, 102, 172)'],    # 1825: Medium blue
        [0.99/5, 'rgb(103, 169, 207)'],  # 1849: Light blue
        
        # Era 2: 1850-1899 (Greens)
        [1.0/5, 'rgb(0, 109, 44)'],      # 1850: Dark green
        [1.5/5, 'rgb(49, 163, 84)'],     # 1875: Medium green
        [1.99/5, 'rgb(161, 217, 155)'],  # 1899: Light green
        
        # Era 3: 1900-1949 (Yellows/Oranges)
        [2.0/5, 'rgb(178, 102, 0)'],     # 1900: Dark orange
        [2.5/5, 'rgb(253, 141, 60)'],    # 1925: Medium orange
        [2.99/5, 'rgb(254, 217, 118)'],  # 1949: Light yellow
        
        # Era 4: 1950-1999 (Reds)
        [3.0/5, 'rgb(127, 0, 0)'],       # 1950: Dark red
        [3.5/5, 'rgb(203, 24, 29)'],     # 1975: Medium red
        [3.99/5, 'rgb(252, 146, 114)'],  # 1999: Light red
        
        # Era 5: 2000+ (Purples)
        [4.0/5, 'rgb(84, 39, 143)'],     # 2000: Dark purple
        [4.5/5, 'rgb(140, 81, 186)'],    # 2025: Medium purple
        [5.0/5, 'rgb(188, 189, 220)'],   # 2050: Light purple
    ]
    
    return colorscale


def create_colorbar_labels(df):
    """
    Create colorbar tick labels showing era ranges and example years.
    """
    
    # Get the actual year range in your data
    min_year = df['year'].min()
    max_year = df['year'].max()
    
    # Determine which eras are present
    eras_present = df['era'].dropna().unique()
    
    # Create labels
    era_labels = {
        1: '1800-1849',
        2: '1850-1899',
        3: '1900-1949',
        4: '1950-1999',
        5: '2000+'
    }
    
    tickvals = []
    ticktext = []
    
    for era in sorted(eras_present):
        # Add tick at middle of era
        tickvals.append(era + 0.5)
        ticktext.append(era_labels[era])
    
    return tickvals, ticktext



def build_reprint_edges(df_articles):
    """
    Build edges showing reprint relationships.
    Edges go from earlier articles to later articles within the same group.
    """
    edges = []
    
    # Group by reprint group
    for group_id, group_df in df_articles.groupby('group_reprint_id'):
        # Sort by date
        group_df = group_df.sort_values('date_dt')
        articles = group_df.to_dict('records')
        
        # Find the original(s)
        originals = [a for a in articles if a['reprint_type'] == 'original']
        
        if originals:
            # Connect original to all reprints
            original = originals[0]
            for article in articles:
                if article['article_id'] != original['article_id']:
                    edges.append({
                        'source': original['article_id'],
                        'target': article['article_id'],
                        'source_lat': original['lat'],
                        'source_lon': original['lon'],
                        'target_lat': article['lat'],
                        'target_lon': article['lon'],
                        'reprint_type': article['reprint_type'],
                        'days_between': (article['date_dt'] - original['date_dt']).days
                    })
        else:
            # No clear original - connect chronologically
            for i in range(len(articles) - 1):
                edges.append({
                    'source': articles[i]['article_id'],
                    'target': articles[i+1]['article_id'],
                    'source_lat': articles[i]['lat'],
                    'source_lon': articles[i]['lon'],
                    'target_lat': articles[i+1]['lat'],
                    'target_lon': articles[i+1]['lon'],
                    'reprint_type': articles[i+1]['reprint_type'],
                    'days_between': (articles[i+1]['date_dt'] - articles[i]['date_dt']).days
                })
    
    return edges

print("✓ Helper functions defined!")

✓ Helper functions defined!


## Step 3: Load and Prepare Data

**MODIFY THESE PARAMETERS:**

In [36]:
# ===== MODIFY THESE =====

csv_file = '/Users/owenmonroe/Desktop/GitHub/lonewoman/complete_metadata_images_tropes_reprints_transcripts.csv'  # Path to your CSV

# Choose 3 reprint groups to visualize (you can add more later)
selected_groups = [
    'TheBostonAtlas1847_reprint',
    'NewYorkHerald1896_reprint',
    'Russell1856_HCF_reprint'
]

# ===== LOAD DATA =====

print(f"Loading data from: {csv_file}")
df = pd.read_csv(csv_file)
print(f"Total records: {len(df)}")

# Get unique articles (not images)
df_articles = df.drop_duplicates(subset='article_id').copy()
print(f"Unique articles: {len(df_articles)}")

# Filter to selected reprint groups
df_articles = df_articles[df_articles['group_reprint_id'].isin(selected_groups)]
print(f"\nArticles in selected groups: {len(df_articles)}")

if len(df_articles) == 0:
    print("\n⚠️ No articles found for selected groups!")
    print("\nAvailable reprint groups:")
    for group in df['group_reprint_id'].unique():
        count = df[df['group_reprint_id'] == group]['article_id'].nunique()
        print(f"  - {group}: {count} articles")
else:
    print("\nSelected groups:")
    for group in selected_groups:
        count = len(df_articles[df_articles['group_reprint_id'] == group])
        if count > 0:
            print(f"  ✓ {group}: {count} articles")

Loading data from: /Users/owenmonroe/Desktop/GitHub/lonewoman/complete_metadata_images_tropes_reprints_transcripts.csv
Total records: 4428
Unique articles: 481

Articles in selected groups: 40

Selected groups:
  ✓ TheBostonAtlas1847_reprint: 14 articles
  ✓ NewYorkHerald1896_reprint: 9 articles
  ✓ Russell1856_HCF_reprint: 17 articles


## Step 4: Geocode Locations

This converts location names (e.g., "Boston, Massachusetts") to coordinates.

**Note:** This may take a minute as it queries OpenStreetMap for each unique location.

In [37]:
# Geocode all locations
location_coords = geocode_locations(df_articles)

# Add coordinates to dataframe
df_articles = add_coordinates_to_df(df_articles, location_coords)

# Parse dates
df_articles = parse_dates_with_era_colors(df_articles)

# Check for missing coordinates
missing_coords = df_articles[df_articles['lat'].isna()]
if len(missing_coords) > 0:
    print(f"\n⚠️ Warning: {len(missing_coords)} articles missing coordinates:")
    for _, row in missing_coords.iterrows():
        print(f"  - {row['article_id']}: {row['publisher_location']}")
    print("\nThese will be excluded from the visualization.")
    df_articles = df_articles[df_articles['lat'].notna()]

print(f"\n✓ Ready to visualize {len(df_articles)} articles!")

Geocoding 32 unique locations...
  ✓ New York, New York: (40.71, -74.01)
  ✓ Honolulu, Hawaii: (21.30, -157.86)
  ✓ Edgefield, South Carolina: (33.76, -81.98)
  ✓ Katanning, Western Australia, Australia: (-33.69, 117.56)
  ✓ Millersburg, Ohio: (40.55, -81.92)
  ✓ San Francisco, California: (37.79, -122.41)
  ✓ Huntington, New York: (40.87, -73.43)
  ✓ Lowell, Massachusetts: (42.64, -71.31)
  ✓ Maryborough, Queensland, Australia: (-25.54, 152.70)
  ✓ Bedford, Massachusetts: (42.49, -71.28)
  ✓ New Orleans, Louisiana: (29.96, -90.07)
  ✓ Oamaru, North Otago, New Zealand: (-45.07, 170.99)
  ✓ Philadelphia, Pennsylvania: (39.95, -75.16)
  ✓ Fort Atkinson, Wisconsin: (42.93, -88.84)
  ✓ Chicago, Illinois: (41.88, -87.62)
  ✓ Sacramento, California: (38.58, -121.49)
  ✓ Santa Barbara, California: (34.42, -119.70)
  ✓ Batavia, New York: (43.00, -78.19)
  ✓ Melbourne, Victoria, Australia: (-37.81, 144.96)
  ✓ Boston, Massachusetts: (42.36, -71.06)
  ✓ Tokomairiro, Otago, New Zealand: (-46.12, 

## Step 5: Build Network Structure

In [38]:
# Build edges showing reprint relationships
edges = build_reprint_edges(df_articles)

print(f"Network structure:")
print(f"  Nodes (articles): {len(df_articles)}")
print(f"  Edges (reprint connections): {len(edges)}")

if len(edges) > 0:
    print(f"\nExample connections:")
    for edge in edges[:3]:
        print(f"  {edge['source']} → {edge['target']} ({edge['days_between']} days, {edge['reprint_type']})")

Network structure:
  Nodes (articles): 40
  Edges (reprint connections): 37

Example connections:
  NewYorkHerald1896 → WeeklyIrishTimes1896 (18 days, direct)
  NewYorkHerald1896 → TheSaintPaulGlobe1896 (36 days, direct)
  NewYorkHerald1896 → TheMorningBulletin1897 (86 days, truncated)


## Step 6: Create Interactive Visualization

This creates the geographic network with:
- Nodes positioned by actual coordinates
- Colors showing time (gradient from blue to red)
- Edges showing reprint flow
- Hover to see details
- Click to highlight lineage (connected via Plotly's built-in interactivity)

In [39]:
# ===== STEP 6: CREATE INTERACTIVE VISUALIZATION =====

# --- Spread out overlapping nodes at the same location ---
# Nodes sharing exact lat/lon get arranged in a small circle so they're
# individually visible and clickable instead of stacked on top of each other.

import math

JITTER_RADIUS = .5  # degrees — visible separation on a world map

# Group articles by their original (lat, lon) and assign offsets
location_groups = df_articles.groupby(['lat', 'lon'])
jitter_lat = {}
jitter_lon = {}

for (lat, lon), group in location_groups:
    n = len(group)
    if n <= 1:
        # Only one node here, no offset needed
        for aid in group['article_id']:
            jitter_lat[aid] = lat
            jitter_lon[aid] = lon
    else:
        # Spread n nodes in a circle around the original point
        for i, aid in enumerate(group['article_id']):
            angle = 2 * math.pi * i / n
            jitter_lat[aid] = lat + JITTER_RADIUS * math.sin(angle)
            jitter_lon[aid] = lon + JITTER_RADIUS * math.cos(angle)

# Apply jittered coordinates
df_articles['lat_display'] = df_articles['article_id'].map(jitter_lat)
df_articles['lon_display'] = df_articles['article_id'].map(jitter_lon)

# Update edge endpoints to match the jittered node positions
for edge in edges:
    edge['source_lat'] = jitter_lat[edge['source']]
    edge['source_lon'] = jitter_lon[edge['source']]
    edge['target_lat'] = jitter_lat[edge['target']]
    edge['target_lon'] = jitter_lon[edge['target']]

# Report which locations were spread out
overlapping = {k: v for k, v in location_groups.groups.items() if len(v) > 1}
print(f"Spread out nodes at {len(overlapping)} overlapping locations:")
for (lat, lon), indices in overlapping.items():
    loc_name = df_articles.loc[indices[0], 'publisher_location']
    print(f"  {loc_name}: {len(indices)} nodes spread in circle (radius={JITTER_RADIUS} deg)")

# Separate originals from reprints for different styling
df_originals = df_articles[df_articles['reprint_type'] == 'original'].copy()
df_reprints = df_articles[df_articles['reprint_type'] != 'original'].copy()

# Create custom colorscale and labels
custom_colorscale = create_custom_colorscale()
colorbar_ticks, colorbar_labels = create_colorbar_labels(df_articles)

# Create figure
fig = go.Figure()

# Add edges (reprint connections)
for edge in edges:
    # Determine line style based on reprint type
    if edge['reprint_type'] == 'paraphrase':
        dash = 'dot'
    elif edge['reprint_type'] == 'truncated':
        dash = 'dash'
    else:
        dash = 'solid'
    
    fig.add_trace(go.Scattergeo(
        lon=[edge['source_lon'], edge['target_lon']],
        lat=[edge['source_lat'], edge['target_lat']],
        mode='lines',
        line=dict(width=2, color='rgba(100, 100, 100, 0.4)', dash=dash),
        hoverinfo='skip',
        showlegend=False
    ))

# Prepare node data
df_articles['hover_text'] = df_articles.apply(
    lambda row: f"<b>{row['title'][:60]}...</b><br>" +
                f"Publication: {row['publication']}<br>" +
                f"Location: {row['publisher_location']}<br>" +
                f"Date: {row['date']}<br>" +
                f"Type: {row['reprint_type']}<br>" +
                f"Group: {row['group_reprint_id']}",
    axis=1
)

df_articles['node_label'] = df_articles.apply(
    lambda row: f"{row['title'][:30]}...<br>{row['publication'][:25]}",
    axis=1
)

# Prepare for separated dataframes
df_originals['hover_text'] = df_originals.apply(
    lambda row: f"<b>{row['title'][:60]}...</b><br>" +
                f"Publication: {row['publication']}<br>" +
                f"Location: {row['publisher_location']}<br>" +
                f"Date: {row['date']}<br>" +
                f"Type: {row['reprint_type']}<br>" +
                f"Group: {row['group_reprint_id']}",
    axis=1
)

df_originals['node_label'] = df_originals.apply(
    lambda row: f"{row['title'][:30]}...<br>{row['publication'][:25]}",
    axis=1
)

df_reprints['hover_text'] = df_reprints.apply(
    lambda row: f"<b>{row['title'][:60]}...</b><br>" +
                f"Publication: {row['publication']}<br>" +
                f"Location: {row['publisher_location']}<br>" +
                f"Date: {row['date']}<br>" +
                f"Type: {row['reprint_type']}<br>" +
                f"Group: {row['group_reprint_id']}",
    axis=1
)

df_reprints['node_label'] = df_reprints.apply(
    lambda row: f"{row['title'][:30]}...<br>{row['publication'][:25]}",
    axis=1
)

# Add ORIGINAL articles (larger, star-shaped) — using jittered coordinates
fig.add_trace(go.Scattergeo(
    lon=df_originals['lon_display'],
    lat=df_originals['lat_display'],
    mode='markers+text',
    name='Original',
    marker=dict(
        size=20,
        color=df_originals['color_value'],
        colorscale=custom_colorscale,
        colorbar=dict(
            title="Era",
            tickvals=colorbar_ticks,
            ticktext=colorbar_labels
        ),
        line=dict(width=3, color='black'),
        symbol='star',
    ),
    text=df_originals['node_label'],
    textposition='top center',
    textfont=dict(size=10, color='black', family='Arial Black'),
    hovertext=df_originals['hover_text'],
    hoverinfo='text',
    customdata=df_originals['article_id']
))

# Add REPRINT articles (smaller circles) — using jittered coordinates
fig.add_trace(go.Scattergeo(
    lon=df_reprints['lon_display'],
    lat=df_reprints['lat_display'],
    mode='markers+text',
    name='Reprints',
    marker=dict(
        size=12,
        color=df_reprints['color_value'],
        colorscale=custom_colorscale,
        showscale=False,
        line=dict(width=2, color='white'),
        symbol='circle',
    ),
    text=df_reprints['node_label'],
    textposition='top center',
    textfont=dict(size=9, color='black'),
    hovertext=df_reprints['hover_text'],
    hoverinfo='text',
    customdata=df_reprints['article_id']
))

# Update layout
fig.update_layout(
    title=dict(
        text="Reprint Network: Movement Across Time and Space<br>" +
             "<sub>Node color = era (1800s=blue/green, 1900s=orange/red) | " +
             "⭐=Original, ⚫=Reprint | " +
             "Lines = reprint connections (solid=direct, dashed=truncated, dotted=paraphrase)</sub>",
        x=0.5,
        xanchor='center'
    ),
    geo=dict(
        projection_type='mercator',
        showland=True,
        landcolor='rgb(243, 243, 243)',
        coastlinecolor='rgb(204, 204, 204)',
        showlakes=True,
        lakecolor='rgb(230, 245, 255)',
        showcountries=True,
        countrycolor='rgb(204, 204, 204)',
        showocean=True,
        oceancolor='rgb(230, 245, 255)',
        # Set the visible lat/lon range to show the whole world
        lataxis=dict(range=[-60, 75]),  # South to North
        lonaxis=dict(range=[-180, 180])  # West to East (full globe)
    ),
    height=900,  # Taller figure
    width=1600,  # Wider figure
    margin=dict(l=0, r=0, t=80, b=0)
)

# Show the figure
fig.show()

print("\n✓ Visualization created!")
print("\nInteraction tips:")
print("  - Hover over nodes to see full article details")
print("  - ⭐ Stars = Original articles")
print("  - ⚫ Circles = Reprints (truncated, paraphrase, etc.)")
print("  - Zoom and pan to explore different regions")
print("  - Use the camera icon to save as image")

Spread out nodes at 4 overlapping locations:
  Honolulu, Hawaii: 2 nodes spread in circle (radius=0.5 deg)
  San Francisco, California: 4 nodes spread in circle (radius=0.5 deg)
  Philadelphia, Pennsylvania: 2 nodes spread in circle (radius=0.5 deg)
  New York, New York: 4 nodes spread in circle (radius=0.5 deg)



✓ Visualization created!

Interaction tips:
  - Hover over nodes to see full article details
  - ⭐ Stars = Original articles
  - ⚫ Circles = Reprints (truncated, paraphrase, etc.)
  - Zoom and pan to explore different regions
  - Use the camera icon to save as image


##  Step 7 Adding Click to Highlight Lineage Functionality

In [40]:
# ===== ADD CLICK-TO-HIGHLIGHT FUNCTIONALITY (FULLY REWRITTEN) =====

import json

# Build mapping of article_id to reprint group
article_to_group = df_articles.set_index('article_id')['group_reprint_id'].to_dict()

# Build mapping of reprint group to all its article IDs
group_to_articles = {}
for group in df_articles['group_reprint_id'].unique():
    group_articles = df_articles[df_articles['group_reprint_id'] == group]['article_id'].tolist()
    group_to_articles[group] = group_articles

# Create edge mapping with indices
edge_mapping = []
for i, edge in enumerate(edges):
    edge_mapping.append({
        'index': i,  # This will be the trace index in the plot
        'source': edge['source'],
        'target': edge['target'],
        'group': article_to_group[edge['source']]
    })

print(f"Debug: Found {len(group_to_articles)} reprint groups")
for group, articles in group_to_articles.items():
    print(f"  - {group}: {len(articles)} articles")

# Count edge traces (they come first in the figure)
num_edge_traces = len(edges)
print(f"Debug: {num_edge_traces} edge traces, then 2 marker traces (originals + reprints)")

# Completely rewritten JavaScript - fixes:
# 1. Robust element finding with polling (not getElementsByClassName('plotly'))
# 2. Waits for plot to be fully initialized before binding events
# 3. Per-point opacity arrays (not per-trace) so individual nodes highlight correctly
# 4. Wrapped in IIFE to avoid global scope pollution
highlight_js = f"""
<script>
(function() {{
    'use strict';
    
    console.log('=== Click-to-Highlight Script Loading ===');
    
    // Data from Python
    var articleToGroup = {json.dumps(article_to_group)};
    var groupToArticles = {json.dumps(group_to_articles)};
    var edgeMapping = {json.dumps(edge_mapping)};
    var numEdgeTraces = {num_edge_traces};
    
    var currentlyHighlighted = null;
    var myPlot = null;
    
    // Store original line styles for each edge trace
    var originalEdgeStyles = [];
    
    function findPlotElement() {{
        // Try multiple selectors in order of specificity
        var el = document.querySelector('.js-plotly-plot') ||
                 document.querySelector('.plotly-graph-div') ||
                 document.querySelector('[class*="plotly"]');
        return el;
    }}
    
    function initClickHandler() {{
        myPlot = findPlotElement();
        
        if (!myPlot) {{
            console.log('Plot element not found yet, retrying in 500ms...');
            setTimeout(initClickHandler, 500);
            return;
        }}
        
        // Check if Plotly has finished initializing (data and layout must exist)
        if (!myPlot.data || !myPlot._fullLayout) {{
            console.log('Plot not fully initialized yet, retrying in 500ms...');
            setTimeout(initClickHandler, 500);
            return;
        }}
        
        console.log('Plot found and ready!');
        console.log('Total traces:', myPlot.data.length);
        
        // Log trace structure for debugging
        for (var i = 0; i < myPlot.data.length; i++) {{
            var t = myPlot.data[i];
            if (t.mode === 'lines') {{
                console.log('Trace', i, ': edge (lines)');
            }} else if (t.customdata) {{
                console.log('Trace', i, ':', t.name, '- customdata:', t.customdata);
            }}
        }}
        
        // Store original edge line styles
        for (var i = 0; i < numEdgeTraces; i++) {{
            var trace = myPlot.data[i];
            if (trace.line) {{
                originalEdgeStyles.push({{
                    color: trace.line.color,
                    width: trace.line.width,
                    dash: trace.line.dash
                }});
            }}
        }}
        
        // Bind the click event
        myPlot.on('plotly_click', handleClick);
        
        // Double-click to reset
        myPlot.on('plotly_doubleclick', function() {{
            console.log('Double-click detected - resetting view');
            resetView();
        }});
        
        console.log('=== Click-to-Highlight Ready! Click any node. ===');
    }}
    
    function handleClick(eventData) {{
        console.log('=== Click detected ===');
        
        var point = eventData.points[0];
        var clickedArticleId = point.customdata;
        
        console.log('Clicked point - customdata:', clickedArticleId, 
                     'traceIndex:', point.curveNumber, 'pointIndex:', point.pointIndex);
        
        if (!clickedArticleId) {{
            console.log('No customdata on this point (probably an edge line). Ignoring.');
            return;
        }}
        
        var reprintGroup = articleToGroup[clickedArticleId];
        if (!reprintGroup) {{
            console.log('Article not found in any group:', clickedArticleId);
            return;
        }}
        
        console.log('Reprint group:', reprintGroup);
        
        // Toggle: clicking same group resets
        if (currentlyHighlighted === reprintGroup) {{
            resetView();
            return;
        }}
        
        currentlyHighlighted = reprintGroup;
        var articlesInGroup = groupToArticles[reprintGroup];
        console.log('Highlighting', articlesInGroup.length, 'articles in group');
        
        // === Update edge traces (per-trace, since each edge = 1 trace) ===
        for (var i = 0; i < numEdgeTraces; i++) {{
            var belongsToGroup = false;
            for (var j = 0; j < edgeMapping.length; j++) {{
                if (edgeMapping[j].index === i && edgeMapping[j].group === reprintGroup) {{
                    belongsToGroup = true;
                    break;
                }}
            }}
            
            if (belongsToGroup) {{
                Plotly.restyle(myPlot, {{
                    'line.color': 'rgba(255, 100, 0, 0.9)',
                    'line.width': 4
                }}, [i]);
            }} else {{
                Plotly.restyle(myPlot, {{
                    'line.color': 'rgba(200, 200, 200, 0.08)',
                    'line.width': 0.5
                }}, [i]);
            }}
        }}
        
        // === Update marker traces (per-POINT opacity arrays) ===
        for (var i = numEdgeTraces; i < myPlot.data.length; i++) {{
            var trace = myPlot.data[i];
            if (!trace.customdata) continue;
            
            var opacities = [];
            var sizes = [];
            var baseSize = (trace.name === 'Original') ? 20 : 12;
            
            for (var j = 0; j < trace.customdata.length; j++) {{
                var articleId = trace.customdata[j];
                if (articlesInGroup.indexOf(articleId) !== -1) {{
                    // This node is in the highlighted group
                    opacities.push(1.0);
                    sizes.push(baseSize * 1.4);
                }} else {{
                    // Fade this node
                    opacities.push(0.12);
                    sizes.push(baseSize * 0.6);
                }}
            }}
            
            Plotly.restyle(myPlot, {{
                'marker.opacity': [opacities],
                'marker.size': [sizes],
                'textfont.color': [opacities.map(function(o) {{ return o > 0.5 ? 'black' : 'rgba(0,0,0,0.08)'; }})]
            }}, [i]);
        }}
        
        showInfoBox(reprintGroup, articlesInGroup.length);
    }}
    
    function resetView() {{
        console.log('=== Resetting view ===');
        currentlyHighlighted = null;
        
        // Reset edge traces to original styles
        for (var i = 0; i < numEdgeTraces; i++) {{
            if (originalEdgeStyles[i]) {{
                Plotly.restyle(myPlot, {{
                    'line.color': originalEdgeStyles[i].color,
                    'line.width': originalEdgeStyles[i].width
                }}, [i]);
            }}
        }}
        
        // Reset marker traces - restore uniform opacity and size
        for (var i = numEdgeTraces; i < myPlot.data.length; i++) {{
            var trace = myPlot.data[i];
            if (!trace.customdata) continue;
            
            var baseSize = (trace.name === 'Original') ? 20 : 12;
            var numPoints = trace.customdata.length;
            
            // Create arrays of uniform values
            var ones = [];
            var defaultSizes = [];
            var blacks = [];
            for (var j = 0; j < numPoints; j++) {{
                ones.push(1.0);
                defaultSizes.push(baseSize);
                blacks.push('black');
            }}
            
            Plotly.restyle(myPlot, {{
                'marker.opacity': [ones],
                'marker.size': [defaultSizes],
                'textfont.color': [blacks]
            }}, [i]);
        }}
        
        // Remove info box
        var infoDiv = document.getElementById('group-info');
        if (infoDiv) infoDiv.remove();
    }}
    
    function showInfoBox(groupName, numArticles) {{
        var infoDiv = document.getElementById('group-info');
        if (!infoDiv) {{
            infoDiv = document.createElement('div');
            infoDiv.id = 'group-info';
            infoDiv.style.cssText = 'position:fixed; top:120px; right:30px; background:rgba(255,255,255,0.95); border:3px solid #ff6600; border-radius:10px; padding:20px; z-index:10000; font-family:Arial,sans-serif; box-shadow:0 6px 12px rgba(0,0,0,0.3); min-width:250px;';
            document.body.appendChild(infoDiv);
        }}
        
        var plural = numArticles > 1 ? 's' : '';
        infoDiv.innerHTML = '<div style="font-size:16px; font-weight:bold; color:#ff6600; margin-bottom:10px;">Highlighted Group</div>' +
            '<div style="font-size:14px; margin-bottom:5px;"><strong>' + groupName + '</strong></div>' +
            '<div style="font-size:13px; color:#666; margin-bottom:15px;">' + numArticles + ' article' + plural + ' in this lineage</div>' +
            '<div style="font-size:11px; color:#999; font-style:italic; border-top:1px solid #ddd; padding-top:10px;">Click another node to switch groups<br>Double-click map to reset view</div>';
    }}
    
    // Start initialization - wait for DOM ready then poll for Plotly
    if (document.readyState === 'complete') {{
        setTimeout(initClickHandler, 300);
    }} else {{
        window.addEventListener('load', function() {{
            setTimeout(initClickHandler, 300);
        }});
    }}
    
}})();
</script>
"""

# Save enhanced HTML
html_string = fig.to_html(include_plotlyjs='cdn')
html_with_js = html_string.replace('</body>', highlight_js + '</body>')

output_filename = 'reprint_network_interactive_enhanced.html'
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(html_with_js)

print("Enhanced HTML with click-to-highlight saved!")
print(f"  File: {output_filename}")
print("\nHow it works:")
print("  1. Open the HTML file in your browser")
print("  2. Click on ANY node (star or circle)")
print("  3. That article's entire reprint lineage will be highlighted")
print("  4. Non-group nodes fade to ~12% opacity")
print("  5. Edges in the lineage turn bright orange and thicken")
print("  6. Click another node to switch groups")
print("  7. Click the same group again or double-click map to reset")

Debug: Found 3 reprint groups
  - TheBostonAtlas1847_reprint: 14 articles
  - Russell1856_HCF_reprint: 17 articles
  - NewYorkHerald1896_reprint: 9 articles
Debug: 37 edge traces, then 2 marker traces (originals + reprints)
Enhanced HTML with click-to-highlight saved!
  File: reprint_network_interactive_enhanced.html

How it works:
  1. Open the HTML file in your browser
  2. Click on ANY node (star or circle)
  3. That article's entire reprint lineage will be highlighted
  4. Non-group nodes fade to ~12% opacity
  5. Edges in the lineage turn bright orange and thicken
  6. Click another node to switch groups
  7. Click the same group again or double-click map to reset


## Step 8: Save the Visualization

In [41]:
# Save as interactive HTML
output_file = 'reprint_network_interactive.html'
fig.write_html(output_file)
print(f"✓ Saved interactive visualization to: {output_file}")
print("\nYou can:")
print("  - Open this HTML file in any browser")
print("  - Embed it in your CollectionBuilder website")
print("  - Share it with collaborators")

# Optional: Save as static image (requires kaleido)
# Uncomment these lines if you want a PNG:
# fig.write_image('reprint_network.png', width=1400, height=900)
# print("✓ Saved static image to: reprint_network.png")

✓ Saved interactive visualization to: reprint_network_interactive.html

You can:
  - Open this HTML file in any browser
  - Embed it in your CollectionBuilder website
  - Share it with collaborators


## Optional: Analyze Network Patterns

Get some statistics about your reprint network:

In [42]:
print("Network Statistics:\n")
print(f"Total articles: {len(df_articles)}")
print(f"Total reprint connections: {len(edges)}")
print(f"\nReprint groups: {df_articles['group_reprint_id'].nunique()}")

print("\nReprint types:")
print(df_articles['reprint_type'].value_counts())

print("\nPublications involved:")
print(df_articles['publication'].value_counts())

print("\nLocations:")
print(df_articles['publisher_location'].value_counts())

if len(edges) > 0:
    edge_df = pd.DataFrame(edges)
    print(f"\nTime between reprints:")
    print(f"  Average: {edge_df['days_between'].mean():.1f} days")
    print(f"  Median: {edge_df['days_between'].median():.1f} days")
    print(f"  Range: {edge_df['days_between'].min()}-{edge_df['days_between'].max()} days")

Network Statistics:

Total articles: 40
Total reprint connections: 37

Reprint groups: 3

Reprint types:
reprint_type
direct        17
truncated     16
paraphrase     4
original       3
Name: count, dtype: int64

Publications involved:
publication
New York Christian Advocate and Journal                          1
Honolulu (Hawaii) Pacific Commercial Advertiser                  1
Batavia (NY) Spirit of the Times                                 1
California Farmer and Journal of Useful Sciences                 1
Melbourne (AUS) Argus                                            1
Boston Atlas                                                     1
Tokomairiro (NZL) Bruce Herald                                   1
Friend                                                           1
Goulburn (AUS) Herald and County of Argyle Advertiser            1
Littell's Living Age                                             1
Memphis (TN) Daily Appeal                                        1
Rockhampton (AU

Step 7 Adding Lineage Highlight Functionality into the HTML

## Optional: Visualize ALL Reprint Groups

Once you're happy with your 3-group prototype, run this cell to see all reprint groups:

In [43]:
# Uncomment and run this cell to visualize ALL reprint groups

# # Load all data
# df_all = pd.read_csv(csv_file)
# df_all_articles = df_all.drop_duplicates(subset='article_id').copy()

# # Filter to only articles that are part of reprint groups
# # (exclude articles with no reprints)
# group_counts = df_all_articles.groupby('group_reprint_id')['article_id'].count()
# groups_with_reprints = group_counts[group_counts > 1].index
# df_all_articles = df_all_articles[df_all_articles['group_reprint_id'].isin(groups_with_reprints)]

# print(f"Total articles in reprint groups: {len(df_all_articles)}")
# print(f"Total reprint groups: {df_all_articles['group_reprint_id'].nunique()}")

# # Then repeat steps 4-6 with df_all_articles instead of df_articles